# 🚀 AlexNet — Notes + Interview
---
> **Simple English** | **Interview Ready** | Year: 2012 | Creator: Alex Krizhevsky

## 📌 What is AlexNet? (Simple English)
- AlexNet **won ImageNet 2012** with 16% error (others had 26%+) — massive gap!
- This **reignited the deep learning revolution**
- First network to use **ReLU** activation (much faster training)
- First to use **Dropout** for regularization
- First to train on **GPU** (2 GTX 580s)
- 8 layers deep (5 Conv + 3 Dense)

## 🔑 AlexNet Innovations
| Innovation | Why it Mattered |
|---|---|
| **ReLU** | Faster training, no vanishing gradient |
| **Dropout** | Prevented overfitting |
| **GPU training** | 10× faster than CPU |
| **Data augmentation** | Flips, crops → more training data |
| **Local Response Normalization** | Lateral inhibition (less used now) |

## 🧱 Architecture
```
Input (227×227×3)
→ Conv(96, 11×11, s=4) → MaxPool
→ Conv(256, 5×5, pad=2) → MaxPool
→ Conv(384, 3×3, pad=1)
→ Conv(384, 3×3, pad=1)
→ Conv(256, 3×3, pad=1) → MaxPool
→ Flatten
→ Dense(4096)+Dropout → Dense(4096)+Dropout → Dense(1000,softmax)
```

In [ ]:
import tensorflow as tf

# ── AlexNet Architecture (adapted for smaller input) ──
def build_alexnet(input_shape=(227,227,3), num_classes=1000):
    model = tf.keras.Sequential([
        # Block 1: Large 11×11 filter at start (original design)
        tf.keras.layers.Conv2D(96, (11,11), strides=4, activation='relu',
                               input_shape=input_shape, padding='valid'),
        tf.keras.layers.MaxPooling2D(3,2),

        # Block 2
        tf.keras.layers.Conv2D(256,(5,5), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(3,2),

        # Block 3-5: Three conv layers back to back (no pool between)
        tf.keras.layers.Conv2D(384,(3,3), activation='relu', padding='same'),
        tf.keras.layers.Conv2D(384,(3,3), activation='relu', padding='same'),
        tf.keras.layers.Conv2D(256,(3,3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(3,2),

        # Classifier
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(4096, activation='relu'),
        tf.keras.layers.Dropout(0.5),   # KEY INNOVATION
        tf.keras.layers.Dense(4096, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ], name='AlexNet')
    return model

alexnet = build_alexnet()
alexnet.summary()
print(f"\nTotal params: {alexnet.count_params():,}  (~60M parameters)")

In [ ]:
# AlexNet on CIFAR-10 (smaller version)
import numpy as np
import matplotlib.pyplot as plt

def build_alexnet_small(num_classes=10):
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same',input_shape=(32,32,3)),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512,activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes,activation='softmax'),
    ], name='AlexNet-Small')

(x_tr,y_tr),(x_te,y_te) = tf.keras.datasets.cifar10.load_data()
x_tr = x_tr.astype('float32')/255.0
x_te = x_te.astype('float32')/255.0

model_small = build_alexnet_small()
model_small.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model_small.fit(x_tr,y_tr,epochs=5,batch_size=128,validation_split=0.1,verbose=1)
_,acc = model_small.evaluate(x_te,y_te,verbose=0)
print(f"\nCIFAR-10 Accuracy: {acc:.4f}")

## 🗣️ Interview Q&A

**Q: What made AlexNet revolutionary?**
> Won ImageNet 2012 with massive gap in accuracy. First to combine: ReLU activation + Dropout + GPU training + data augmentation — all in one deep CNN.

**Q: Why is ReLU important in AlexNet?**
> Previous networks used sigmoid/tanh which suffer from vanishing gradients. ReLU = max(0,x) — doesn't saturate for positive values, gradients flow easily, 6× faster convergence.

**Q: What is Dropout?**
> Randomly zeros out neurons during training (e.g. 50%). Prevents co-adaptation — neurons can't rely on specific others. Acts like ensemble of many networks. Reduces overfitting.

**Q: What was the significance of GPU training?**
> AlexNet was split across 2 GPUs — enabled much larger networks and faster training. Showed that GPUs are essential for deep learning — GPU became the standard hardware.